# 06 - Predict sentiment on real comments with all three models

Runs TF-IDF, ATAE-LSTM and DeBERTa on every (comment, aspect) row found by notebook 05 (extract aspects), then shows where the models agree and disagree.

**Input:** `data/comments/comments_aspects.csv` and the trained models from notebooks 01, 02 and 03. A model that is missing is skipped.
**Output (in `results/`):**
- `tfidf_predictions.csv`, `atae_predictions.csv`, `deberta_predictions.csv`: one file per model, with a predicted polarity, a confidence and all class scores per row (the layout the app reads)
- `all_model_predictions.csv`: the three models side by side, one row per (comment, aspect)
- `runtime_agreement.csv`: the same rows with an agreement flag and the odd model out

Real comments have no true labels, so agreement shows where the models differ, not which one is right. The model to trust is the one that wins on the test set in notebook 04.

The prediction code for each model lives in `src/opinion_owl/inference.py`, shared with notebook 04.

In [ ]:
import json

import pandas as pd

from opinion_owl.config import COMMENTS_DIR, RESULTS_DIR
from opinion_owl.inference import MODELS, model_is_trained

INPUT_PATH = COMMENTS_DIR / "comments_aspects.csv"
ALL_PREDICTIONS_PATH = RESULTS_DIR / "all_model_predictions.csv"
AGREEMENT_PATH = RESULTS_DIR / "runtime_agreement.csv"
BEST_MODEL_PATH = RESULTS_DIR / "best_model.json"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"{INPUT_PATH} not found. Run notebook 05 (extract aspects) first.")

## 1. Load and clean the aspect rows

In [ ]:
aspect_data = pd.read_csv(INPUT_PATH)
print("Rows:", len(aspect_data), "| Columns:", aspect_data.columns.tolist())

missing_columns = {"comment_id", "text", "aspect_term"} - set(aspect_data.columns)
if missing_columns:
    raise ValueError("comments_aspects.csv is missing columns: " + ", ".join(sorted(missing_columns)))

aspect_data = aspect_data.dropna(subset=["comment_id", "text", "aspect_term"]).copy()
aspect_data["text"] = aspect_data["text"].astype(str).str.strip()
aspect_data["aspect_term"] = aspect_data["aspect_term"].astype(str).str.strip().str.lower()
aspect_data = aspect_data[aspect_data["text"].ne("") & aspect_data["aspect_term"].ne("")].reset_index(drop=True)

if "raw_aspect" not in aspect_data.columns:
    aspect_data["raw_aspect"] = ""

print("Usable rows:", len(aspect_data))

## 2. Predict with each model

In [ ]:
# Same columns the three separate prediction notebooks used to write.
OUTPUT_COLUMNS = [
    "comment_id", "text", "raw_aspect", "aspect_term",
    "predicted_polarity", "confidence", "all_class_scores", "model",
]

predictions = {}   # model name -> prediction table, in the same row order as aspect_data

for model_name, (runner, required_files, source_notebook, stem) in MODELS.items():
    output_path = RESULTS_DIR / f"{stem}_predictions.csv"
    if not model_is_trained(model_name):
        output_path.unlink(missing_ok=True)   # never leave an old file from an earlier model
        print(f"[skip] {model_name}: model not found. Run {source_notebook} first.")
        continue

    print(f"Predicting {len(aspect_data)} rows with {model_name}...")
    result = runner(aspect_data)
    table = aspect_data[["comment_id", "text", "raw_aspect", "aspect_term"]].join(result)
    table[OUTPUT_COLUMNS].to_csv(output_path, index=False)
    predictions[model_name] = table
    print(f"  saved {len(table)} rows to {output_path.name}")

if not predictions:
    raise RuntimeError("No trained models found. Run notebooks 01, 02 and 03 first.")

## 3. Side by side

In [ ]:
# One row per (comment, aspect), with each model's answer next to the others.
side_by_side = aspect_data[["comment_id", "aspect_term", "text"]].copy()
for model_name, table in predictions.items():
    side_by_side[f"{model_name}_polarity"] = table["predicted_polarity"]
    side_by_side[f"{model_name}_confidence"] = table["confidence"]

side_by_side.to_csv(ALL_PREDICTIONS_PATH, index=False)
print("Saved:", ALL_PREDICTIONS_PATH.name)

polarity_columns = [f"{m}_polarity" for m in predictions]
polarity_mix = pd.DataFrame({m: side_by_side[f"{m}_polarity"].value_counts() for m in predictions}).fillna(0).astype(int)
print("How many rows each model put in each class:")
polarity_mix

In [ ]:
# The least certain predictions for each model: the rows most worth reading by hand.
# TF-IDF confidence comes from a softmax over decision scores, so compare it across rows of
# the same model, not across models.
for model_name, table in predictions.items():
    print("=" * 70)
    print(model_name, "- 10 least confident rows")
    display(
        table.sort_values("confidence")[["comment_id", "aspect_term", "predicted_polarity", "confidence"]].head(10)
    )

## 4. Do the models agree?

In [ ]:
model_names = list(predictions)

if len(model_names) < 2:
    print("Only one model has predictions, so there is nothing to compare. Train the other models first.")
else:
    labels = side_by_side[polarity_columns]
    side_by_side["all_agree"] = labels.nunique(axis=1).eq(1)

    def odd_one_out(row):
        """With three models and a 2-vs-1 split, the model that disagrees with the other two."""
        values = row[polarity_columns].tolist()
        counts = pd.Series(values).value_counts()
        if len(model_names) == 3 and len(counts) == 2:
            lone_label = counts.index[counts == 1][0]
            return model_names[values.index(lone_label)]
        return ""

    side_by_side["odd_one_out"] = side_by_side.apply(odd_one_out, axis=1)

    print(f"Rows: {len(side_by_side)}")
    print(f"All {len(model_names)} models agree: {side_by_side['all_agree'].mean() * 100:.1f}%")

    # How often each pair of models gives the same answer.
    pairwise = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
    for a in model_names:
        for b in model_names:
            pairwise.loc[a, b] = (side_by_side[f"{a}_polarity"] == side_by_side[f"{b}_polarity"]).mean() * 100
    print("\nPairwise agreement (%):")
    display(pairwise.round(1))

    if len(model_names) == 3:
        print("\nWhen two models agree and one does not, the one that disagrees:")
        display(side_by_side["odd_one_out"].replace("", pd.NA).value_counts().rename("rows").to_frame())

In [ ]:
if len(model_names) >= 2:
    # Sort disagreements by how sure the reference model was. The reference is the winner of
    # notebook 04 if best_model.json exists, otherwise DeBERTa, otherwise the first model.
    reference = None
    if BEST_MODEL_PATH.exists():
        with open(BEST_MODEL_PATH) as f:
            reference = json.load(f).get("best_model")
    if reference not in predictions:
        reference = "DeBERTa-v3" if "DeBERTa-v3" in predictions else model_names[0]
    print("Reference model for sorting:", reference)

    disagreements = side_by_side[~side_by_side["all_agree"]].sort_values(
        f"{reference}_confidence", ascending=False
    )
    print("Disagreements:", len(disagreements), "of", len(side_by_side))

    show_columns = ["comment_id", "aspect_term"] + polarity_columns + [f"{reference}_confidence", "odd_one_out"]
    display(disagreements[show_columns].head(30))

    side_by_side.to_csv(AGREEMENT_PATH, index=False)
    print("Saved:", AGREEMENT_PATH.name)